In [ ]:
!pip install MEDS-Inspect

In [ ]:
MEDS_Inspect_cache "/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"

In [ ]:
!MEDS_Inspect port=8052 +initial_path="/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"

# Imports

In [ ]:
import os
import pandas as pd
import subprocess
import numpy as np
import hail as hl
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from datetime import datetime

In [ ]:
import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))
os.environ["WORKSPACE_CDR"] = "wb-silky-artichoke-2408.C2025Q4R6"

In [ ]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 300g "
    "--conf spark.driver.maxResultSize=16g "
    "--conf spark.default.parallelism=64 "
    "--conf spark.sql.shuffle.partitions=256 "
    "pyspark-shell"
)

import hail as hl

hl.init(
    master="local[64]",
    idempotent=True,
    default_reference = "GRCh38"
)

In [ ]:
hl.stop()
hl.init(default_reference = "GRCh38")

# Clinical Data

In [ ]:
dataset_08947253_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id"""

dataset_08947253_person_df = pd.read_gbq(
    dataset_08947253_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_08947253_person_df

In [ ]:
DATA_BUCKET = '/home/jupyter/workspace/data_bucket'
GENETIC_FOLDER = f'{DATA_BUCKET}/v9_gen_data'
dataset_08947253_person_df.to_csv(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv", sep = "\t", index=False)
dataset_hl = (hl.import_table(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv",
                              types={'person_id':hl.tstr},
                              impute=True,
                              key='person_id')
             )

# Genetic Data

In [ ]:
vat_path = "/home/jupyter/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz"
vat_path
vat_table = hl.import_table(
    vat_path, 
    force=True, 
    quote='"', 
    delimiter="\t", 
    force_bgz=True, 
    types={"position": hl.tint, "contig": hl.tstr, "ref_allele": hl.tstr, "alt_allele": hl.tstr}
)
vat_table.describe()

In [ ]:
transcripts_of_interest = [
    "ENST00000279146", "ENST00000389048", "ENST00000257430", "ENST00000675843", "ENST00000307078",
    "ENST00000460680", "ENST00000260947", "ENST00000355112", "ENST00000372037", "ENST00000357654",
    "ENST00000380152", "ENST00000259008", "ENST00000639785", "ENST00000367435", "ENST00000261769",
    "ENST00000257904", "ENST00000228872", "ENST00000313407", "ENST00000440480", "ENST00000304494",
    "ENST00000579755", "ENST00000498907", "ENST00000404276", "ENST00000302763", "ENST00000343455",
    "ENST00000325385", "ENST00000275493", "ENST00000263735", "ENST00000366560", "ENST00000285071",
    "ENST00000341105", "ENST00000487848", "ENST00000370818", "ENST00000651081", "ENST00000300177",
    "ENST00000651154", "ENST00000290295", "ENST00000610977", "ENST00000311189", "ENST00000417302",
    "ENST00000288135", "ENST00000358664", "ENST00000555147", "ENST00000450708", "ENST00000397752",
    "ENST00000394351", "ENST00000352241", "ENST00000231790", "ENST00000233146", "ENST00000265081",
    "ENST00000234420", "ENST00000456914", "ENST00000710952", "ENST00000265433", "ENST00000358273",
    "ENST00000338641", "ENST00000651570", "ENST00000261584", "ENST00000257290", "ENST00000226382",
    "ENST00000265849", "ENST00000440232", "ENST00000320574", "ENST00000357628", "ENST00000589228",
    "ENST00000331920", "ENST00000437951", "ENST00000644628", "ENST00000371953", "ENST00000378823",
    "ENST00000337432", "ENST00000345365", "ENST00000267163", "ENST00000617875", "ENST00000355710",
    "ENST00000675419", "ENST00000264932", "ENST00000301761", "ENST00000375499", "ENST00000367975",
    "ENST00000375549", "ENST00000342988", "ENST00000344626", "ENST00000646693", "ENST00000618915",
    "ENST00000644036", "ENST00000348513", "ENST00000326873", "ENST00000369902", "ENST00000310581",
    "ENST00000258439", "ENST00000269305", "ENST00000298552", "ENST00000219476", "ENST00000256474",
    "ENST00000298139", "ENST00000452863"
]
transcript_vat_table = vat_table.filter(
    hl.any(lambda t: vat_table.transcript.startswith(t), hl.literal(transcripts_of_interest))
)

In [ ]:
lst = [
    'pathogenic', 
    'likely pathogenic', 
    'likely pathogenic, pathogenic',  
    'likely pathogenic, affects',
    'likely pathogenic, association',
    'likely pathogenic, drug response',
    'likely pathogenic, drug response, not provided',
    'likely pathogenic, not provided',
    'likely pathogenic, other',
    'likely pathogenic, pathogenic, affects',
    'likely pathogenic, pathogenic, association',
    'likely pathogenic, pathogenic, drug response',
    'likely pathogenic, pathogenic, drug response, not provided',
    'likely pathogenic, pathogenic, drug response, other',
    'likely pathogenic, pathogenic, drug response, protective',
    'likely pathogenic, pathogenic, drug response, risk factor',
    'likely pathogenic, pathogenic, not provided',
    'likely pathogenic, pathogenic, other',
    'likely pathogenic, pathogenic, protective',
    'likely pathogenic, pathogenic, risk factor',
    'likely pathogenic, pathogenic, risk factor, not provided',
    'likely pathogenic, risk factor',
    'likely risk allele',
    'pathogenic, association',
    'pathogenic, association, protective',
    'pathogenic, confers sensitivity',
    'pathogenic, drug response',
    'pathogenic, drug response, not provided',
    'pathogenic, drug response, risk factor',
    'pathogenic, drug response, risk factor, protective',
    'pathogenic, not provided',
    'pathogenic, other',
    'pathogenic, protective',
    'pathogenic, protective, other',
    'pathogenic, risk factor',
    'pathogenic, risk factor, not provided',
    'pathogenic, risk factor, other',
    'pathogenic, risk factor, protective',
    'risk factor'
]

# Filter vat_table to only contain Clinvar classification in lst
filtered_vat_table = transcript_vat_table.filter(hl.set(lst).contains(transcript_vat_table.clinvar_classification))

# Create locus and alleles to match MatrixTable key structure
filtered_vat_table = filtered_vat_table.annotate(
    locus=hl.locus(filtered_vat_table.contig, filtered_vat_table.position, reference_genome='GRCh38'),
    alleles=[filtered_vat_table.ref_allele, filtered_vat_table.alt_allele]
)
filtered_vat_table = filtered_vat_table.key_by('locus', 'alleles')

# List of consequences to exclude
exclude_consequences = ['downstream_gene_variant', 'upstream_gene_variant']

# Filter the table to exclude rows with these consequences
filtered_vat_table = filtered_vat_table.filter(
    ~hl.set(exclude_consequences).contains(filtered_vat_table.consequence)
)


In [ ]:
import hail as hl

mt_wgs_clinvar_path = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/clinvar/splitMT/hail.mt"
)

genes_of_interest = {
    "AIP", "ALK", "APC", "ATM", "AXIN2", "BAP1", "BARD1", "BLM",
    "BMPR1A", "BRCA1", "BRCA2", "BRIP1", "CASR", "CDC73", "CDH1",
    "CDK4", "CDKN1B", "CDKN1C", "CDKN2A", "CEBPA", "CHEK2",
    "CTNNA1", "DICER1", "DIS3L2", "EGFR", "EPCAM", "FH", "FLCN",
    "GATA2", "GPC3", "GREM1", "HOXB13", "HRAS", "KIT", "MAX",
    "MC1R", "MEN1", "MET", "MITF", "MLH1", "MSH2", "MSH3", "MSH6",
    "MUTYH", "NBN", "NF1", "NF2", "NTHL1", "PALB2", "PDGFRA",
    "PHOX2B", "PMS2", "POLD1", "POLE", "POT1", "PRKAR1A", "PTCH1",
    "PTEN", "RAD50", "RAD51C", "RAD51D", "RB1", "RECQL4", "RET",
    "RUNX1", "SDHA", "SDHAF2", "SDHB", "SDHC", "SDHD", "SMAD4",
    "SMARCA4", "SMARCB1", "SMARCE1", "STK11", "SUFU", "TERC", "TERT",
    "TMEM127", "TP53", "TSC1", "TSC2", "VHL", "WRN", "WT1"
}

# Read MatrixTable
mt = hl.read_matrix_table(mt_wgs_clinvar_path)

# Keep only cohort samples and selected variants
mt = mt.semi_join_cols(dataset_hl)
mt = mt.semi_join_rows(filtered_vat_table)

# Add sample metadata
mt = mt.annotate_cols(
    metadata=dataset_hl[mt.s]
)

# Add only the row annotation actually needed
variant_annotations = filtered_vat_table.select("gene_symbol")

mt = mt.annotate_rows(
    gene_symbol=variant_annotations[mt.locus, mt.alleles].gene_symbol
)

# Keep selected genes
genes_literal = hl.literal(genes_of_interest)

mt = mt.filter_rows(
    hl.is_defined(mt.gene_symbol) &
    genes_literal.contains(mt.gene_symbol)
)

# Filter non-reference entries BEFORE calling entries()
mt = mt.filter_entries(
    hl.is_defined(mt.GT) &
    mt.GT.is_non_ref()
)

# Write a filtered MatrixTable checkpoint
filtered_mt_path = f"{GENETIC_FOLDER}/filtered_nonref.mt"
mt = mt.checkpoint(
    filtered_mt_path,
    overwrite=True
)

In [ ]:
# Load the existing filtered MatrixTable
mt = hl.read_matrix_table(f"{GENETIC_FOLDER}/filtered_nonref.mt")

# Restore all VAT annotation fields
mt = mt.annotate_rows(
    annotations=filtered_vat_table[mt.locus, mt.alleles]
)

# Restore the V8 column, if needed
mt = mt.annotate_entries(
    has_variant=hl.is_defined(mt.GT) & mt.GT.is_non_ref()
)

# Recreate only the entries table
entries_table = mt.entries()

# Convert and save
entries_table_df = entries_table.to_pandas()

entries_table_df.to_csv(
    f"{GENETIC_FOLDER}/entries_table_full_v9.csv",
    index=False
)

print(entries_table_df.shape)

In [ ]:
mt = hl.read_matrix_table(filtered_mt_path)
# Extract only needed fields
entries_table = mt.entries()

# Write distributed Hail Table first
entries_ht_path = f"{GENETIC_FOLDER}/entries_table_v9.ht"

entries_table = entries_table.checkpoint(
    entries_ht_path,
    overwrite=True
)

# Export as multiple compressed TSV shards
entries_table.export(
    f"{GENETIC_FOLDER}/entries_table_v9.tsv.bgz",
    parallel="header_per_shard"
)

In [ ]:
entries_ht_path = f"{GENETIC_FOLDER}/entries_table_v9.ht"
entries_table = hl.read_table(entries_ht_path)
entries_table_df = entries_table.to_pandas()

entries_table_df.to_csv(f'{GENETIC_FOLDER}/entries_table_full_v9.csv', index=False)

In [ ]:
entries = pd.read_csv(f"{GENETIC_FOLDER}/entries_table_full_v9.csv")

In [ ]:
entries[entries['gene_symbol']=='SDHB']

In [ ]:
mt_wgs_clinvar_path = '/home/jupyter/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/clinvar/splitMT/hail.mt'

mt = hl.read_matrix_table(mt_wgs_clinvar_path)
mt.describe()

In [ ]:
# Select only the samples in dataset_hl
mt_sub = mt.semi_join_cols(dataset_hl)
# Select only the variants in filtered_vat_table
mt_sub = mt_sub.semi_join_rows(filtered_vat_table)
mt_sub.describe()

# Annotate MatrixTable columns with metadata
mt_sub = mt_sub.annotate_cols(metadata=dataset_hl[mt_sub.s])
# Annotate MatrixTable rows with variant annotations
mt_sub = mt_sub.annotate_rows(annotations=filtered_vat_table[mt_sub.locus, mt_sub.alleles])
# Extract the column fields (metadata and sample ID 's') into a Hail Table
metadata_table = mt_sub.cols()

# Convert the Hail Table to a Pandas DataFrame
metadata_df = metadata_table.to_pandas()

# Save the DataFrame to a CSV file
metadata_df.to_csv(f'{GENETIC_FOLDER}/metadata_v9.csv', index=False)

In [ ]:
# Kind of redundant though
# Filter the MatrixTable to specific genes
genes_of_interest = ['AIP', 'ALK', 'APC', 'ATM', 'AXIN2', 'BAP1', 'BARD1', 'BLM', 'BMPR1A', 'BRCA1', 'BRCA2', 
                     'BRIP1', 'CASR', 'CDC73', 'CDH1', 'CDK4', 'CDKN1B', 'CDKN1C', 'CDKN2A', 'CEBPA', 'CHEK2', 
                     'CTNNA1', 'DICER1', 'DIS3L2', 'EGFR', 'EPCAM', 'FH', 'FLCN', 'GATA2', 'GPC3', 'GREM1', 
                     'HOXB13', 'HRAS', 'KIT', 'MAX', 'MC1R', 'MEN1', 'MET', 'MITF', 'MLH1', 'MSH2', 'MSH3', 
                     'MSH6', 'MUTYH', 'NBN', 'NF1', 'NF2', 'NTHL1', 'PALB2', 'PDGFRA', 'PHOX2B', 'PMS2', 
                     'POLD1', 'POLE', 'POT1', 'PRKAR1A', 'PTCH1', 'PTEN', 'RAD50', 'RAD51C', 'RAD51D', 'RB1', 
                     'RECQL4', 'RET', 'RUNX1', 'SDHA', 'SDHAF2', 'SDHB', 'SDHC', 'SDHD', 'SMAD4', 'SMARCA4', 
                     'SMARCB1', 'SMARCE1', 'STK11', 'SUFU', 'TERC', 'TERT', 'TMEM127', 'TP53', 'TSC1', 'TSC2', 
                     'VHL', 'WRN', 'WT1']

mt_filtered = mt_sub.filter_rows(hl.literal(genes_of_interest).contains(mt_sub.annotations.gene_symbol))
mt_filtered = mt_filtered.annotate_entries(has_variant=mt_filtered.GT.is_non_ref())
mt_filtered.describe()

In [ ]:
entries_table = mt_filtered.entries()
# Filter entries to include only non-reference variants
entries_table = entries_table.filter(entries_table.has_variant)
entries_table.describe()

In [ ]:
import pandas as pd
entries_table_df = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/entries_table_full_v8.csv')

# entries_table_df = entries_table_df[entries_table_df['annotations.gene_symbol'] != 'MC1R']
# entries_table_df = entries_table_df[
#     ~(
#         ((entries_table_df['annotations.gene_symbol'] == 'EPCAM') & 
#          (entries_table_df['annotations.variant_type'] == 'deletion')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'PDGFRA') & 
#          (entries_table_df['annotations.vid'] == '4-54281602-C-T')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'EGFR') & 
#          (entries_table_df['annotations.vid'] == '7-55173126-T-C')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'RET') & 
#          (entries_table_df['annotations.vid'] == '10-43100576-C-T')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'RET') & 
#          (entries_table_df['annotations.vid'] == '10-43106497-G-A')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'TSC1') & 
#          (entries_table_df['annotations.vid'] == '9-132921940-T-G')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'TMEM127') & 
#          (entries_table_df['annotations.vid'] == '2-96265399-G-A')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
#          (entries_table_df['annotations.vid'] == '5-1293489-C-G')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
#          (entries_table_df['annotations.vid'] == '5-1268581-G-A')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
#          (entries_table_df['annotations.vid'] == '5-1254461-C-T')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'POLE') & 
#          (entries_table_df['annotations.vid'] == '12-132680048-T-C')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'POLE') & 
#          (entries_table_df['annotations.vid'] == '12-132677577-C-T')) |  
#         ((entries_table_df['annotations.gene_symbol'] == 'ALK') & 
#          (entries_table_df['annotations.vid'] == '2-29220747-C-T'))
#     )
# ]
print(entries_table_df.shape)
entries_table_df.s.nunique()

In [ ]:
entries_table_df.to_csv(f'/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_full_v9.csv', index=False)

# Analysis

In [ ]:
entries_v9 = pd.read_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_full_v9.csv')
entries_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/entries_table_full_v8.csv')

In [ ]:
entries_v9.columns.values